# 🚗 Car Price Prediction Model (CarDekho Dataset)

This notebook demonstrates an end-to-end Machine Learning pipeline to predict the selling price of used cars based on historical sales data from **CarDekho**.

---
### 📌 Machine Learning Workflow
```text
              CAR DATASET
                   ↓
           Data Preprocessing
                   ↓
            Data Cleaning
                   ↓
          Feature Selection
                   ↓
          Train/Test Split
                   ↓
        Machine Learning Model
        (Linear Regression vs Random Forest)
                   ↓
             Model Training & Tuning
                   ↓
          Model Evaluation (RMSE, R²)
                   ↓
          Sample Prediction
```


## 1. Import Libraries
We import essential packages for data manipulation, visualization, and machine learning.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print('All libraries imported successfully!')

## 2. Load Dataset
We load the CarDekho dataset (`car_data.csv`) and examine the initial records.

In [2]:
# Load data
df = pd.read_csv('../data/car_data.csv')
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## 3. Exploratory Data Analysis (EDA)
Let us inspect data types, missing values, summary statistics, and categorical distributions.

In [3]:
# Check dataset info and missing values
print('Missing Values per Column:')
print(df.isnull().sum())
print('\nDataset Summary Statistics:')
df.describe()

In [4]:
# Inspect unique categories
print('Fuel Types:', df['Fuel_Type'].unique())
print('Seller Types:', df['Seller_Type'].unique())
print('Transmission Types:', df['Transmission'].unique())
print('Owner Values:', df['Owner'].unique())

### Visualizing Target Price Distributions

In [5]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['Selling_Price'], kde=True, color='#2563eb', bins=25)
plt.title('Distribution of Selling Price (Target)', fontsize=12, fontweight='bold')
plt.xlabel('Selling Price (Lakhs)')

plt.subplot(1, 2, 2)
sns.histplot(df['Present_Price'], kde=True, color='#059669', bins=25)
plt.title('Distribution of Showroom / Present Price', fontsize=12, fontweight='bold')
plt.xlabel('Present Price (Lakhs)')

plt.tight_layout()
plt.show()

## 4. Feature Engineering & Preprocessing
1. Instead of raw `Year`, we calculate vehicle age: `Car_Age = 2024 - Year`.
2. Drop `Car_Name` and `Year`.
3. Apply One-Hot Encoding (`pd.get_dummies(drop_first=True)`).

In [6]:
# Create feature Car_Age
CURRENT_YEAR = 2024
final_df = df.copy()
final_df['Car_Age'] = CURRENT_YEAR - final_df['Year']
final_df.drop(columns=['Car_Name', 'Year'], inplace=True)

# One-Hot Encoding
final_df = pd.get_dummies(final_df, drop_first=True, dtype=int)
print(f'Final Processed Shape: {final_df.shape}')
final_df.head()

### Feature Correlation Matrix

In [7]:
plt.figure(figsize=(10, 8))
sns.heatmap(final_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.show()

## 5. Train-Test Split (80% Train, 20% Test)
We separate features (`X`) and target price (`y`).

In [8]:
X = final_df.drop(columns=['Selling_Price'])
y = final_df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training set shape: {X_train.shape}, Test set shape: {X_test.shape}')

## 6. Model 1: Linear Regression (Baseline)
We train an ordinary least squares Linear Regression model.

In [9]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print('=== Linear Regression Evaluation ===')
print(f'MAE:      {mae_lr:.4f} Lakhs')
print(f'RMSE:     {rmse_lr:.4f} Lakhs')
print(f'R2 Score: {r2_lr:.4f}')

## 7. Model 2: Random Forest Regressor (with Hyperparameter Tuning)
We tune an ensemble of Decision Trees using `RandomizedSearchCV` to optimize depth, estimators, and leaf splits.

In [10]:
param_grid = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_features': [1.0, 'sqrt', 'log2'],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestRegressor(random_state=42)
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    scoring='neg_mean_squared_error',
    n_iter=25,
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)
rf_random.fit(X_train, y_train)

best_rf = rf_random.best_estimator_
print('Best Hyperparameters:', rf_random.best_params_)

In [11]:
y_pred_rf = best_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print('=== Random Forest Regressor Evaluation ===')
print(f'MAE:      {mae_rf:.4f} Lakhs')
print(f'RMSE:     {rmse_rf:.4f} Lakhs')
print(f'R2 Score: {r2_rf:.4f}')

## 8. Model Comparison & Visualizations

In [12]:
results_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor (Tuned)'],
    'MAE (Lakhs)': [mae_lr, mae_rf],
    'RMSE (Lakhs)': [rmse_lr, rmse_rf],
    'R2 Score': [r2_lr, r2_rf]
})
results_df

In [13]:
# Actual vs Predicted Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, color='#2563eb', alpha=0.7, edgecolors='k', s=60, label='Predicted')
min_v = min(y_test.min(), y_pred_rf.min())
max_v = max(y_test.max(), y_pred_rf.max())
plt.plot([min_v, max_v], [min_v, max_v], color='#dc2626', linestyle='--', linewidth=2, label='Perfect Fit (y=x)')
plt.title('Actual vs Predicted Selling Prices (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Actual Price (Lakhs)')
plt.ylabel('Predicted Price (Lakhs)')
plt.legend()
plt.show()

In [14]:
# Feature Importance Bar Chart
feat_importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
feat_importances.plot(kind='barh', color='#3b82f6', edgecolor='#1d4ed8')
plt.title('Feature Importances in Car Price Prediction', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance')
plt.show()

## 9. Sample Prediction
Let us test a realistic sample car:
- **Car**: Hyundai i20
- **Year**: 2019 (Age = 5 years)
- **Showroom Price**: 7.5 Lakhs
- **Kilometers Driven**: 35,000 km
- **Fuel**: Petrol
- **Seller**: Dealer
- **Transmission**: Manual
- **Owners**: 0 (First Owner)

In [15]:
sample_test = pd.DataFrame([{
    'Present_Price': 7.5,
    'Kms_Driven': 35000,
    'Owner': 0,
    'Car_Age': 2024 - 2019,
    'Fuel_Type_Diesel': 0,
    'Fuel_Type_Petrol': 1,
    'Seller_Type_Individual': 0,
    'Transmission_Manual': 1
}])[X.columns]

predicted_val = best_rf.predict(sample_test)[0]
print(f'Estimated Selling Price: {predicted_val:.2f} Lakhs (Rs. {int(predicted_val * 100000):,})')
print(f'Estimated Depreciation: {((7.5 - predicted_val) / 7.5)*100:.1f}%')